# 06C — Graph Wiring: Pluggable Index Scores on Activation and Embedding Manifolds

---

## Conceptual framework

ArrowSpace is a **generic graph-wiring algorithm** structured as a modular pipeline:

```
Clustering → Sampling / Compression → Graph Laplacian → IndexScorePhase
```

The terminal phase — `IndexScorePhase` — is **pluggable**. By hot-swapping the scoring
function injected into the pipeline, the same graph structure supports five distinct
algorithmic capabilities:

| Capability | `IndexScorePhase` | Core computation |
|---|---|---|
| **Search** | `TauModeScore` | Rayleigh quotient + Dirichlet dispersion → synthetic λ |
| **Analysis** | `FeatureSpectralScore` | F×F feature-to-feature Laplacian, spectral gaps |
| **Clustering** | `FiedlerVectorScore` | 2nd smallest eigenvector (Fiedler) of normalised Laplacian |
| **Classification** | `BoundaryEnergyScore` | Discrete Dirichlet energy relative to class centroids |
| **Compression** | `EnergyDiffusionScore` | Diffuse graph, optical binning, evict low-λ redundant items |

This notebook instantiates a subset of these phases as **`build_score_phase(name, X, ...)`**
calls, compares them on the activation and embedding manifolds built in `06_B`, and
produces a final capability mapping.

> **Dependency**: this notebook requires `X_pass`, `X_arrow`, `act_matrix`, `words`, `labels`
> and `GRAPH_PARAMS` from `06_B`. Run `06_B` first, or re-run the compact setup cell below
> to rebuild those artefacts independently.

---
## 0 · Shared setup (self-contained)

This cell reproduces the minimal artefacts from `06_B` needed by §§1–5 below:
`X_pass`, `X_arrow`, `act_matrix`, `words`, `labels`, `GRAPH_PARAMS`.
It is intentionally compact — refer to `06_B` for the full mechanistic-probing
commentary.

In [ ]:
# ── stdlib / data ──────────────────────────────────────────────────────────
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

# ── ML / embedding ─────────────────────────────────────────────────────────
import torch
from sentence_transformers import SentenceTransformer

# ── ArrowSpace ─────────────────────────────────────────────────────────────
from arrowspace import ArrowSpaceBuilder

# ── Analysis / viz ─────────────────────────────────────────────────────────
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.neighbors import KernelDensity
from sklearn.metrics import pairwise_distances
from scipy.stats import pearsonr
from scipy.spatial.distance import cdist
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# ── Hyper-parameters ───────────────────────────────────────────────────────
ARROW_MAG   = 1.12
KNN_K       = 12
ALPHA_STEPS = 11
TOP_K_PCT   = 0.15
OUTPUT_DIR  = Path("output__06")
OUTPUT_DIR.mkdir(exist_ok=True)

GRAPH_PARAMS = {'eps': 1.9, 'k': KNN_K, 'topk': 10, 'p': 2.0, 'sigma': None}

FIELD_COLOURS = {
    "FOOD":    "#f58231", "SCIENCE": "#3cb44b",
    "TOOL":    "#4363d8", "COLOUR":  "#f032e6",
}

SEMANTIC_FIELDS = {
    "FOOD": [
        "bread", "rice", "soup", "cake", "pizza", "pasta", "salad",
        "curry", "cheese", "butter", "cream", "jam", "honey",
        "chocolate", "coffee", "tea", "wine", "beer", "milk", "sugar"
    ],
    "SCIENCE": [
        "atom", "electron", "proton", "neutron", "photon", "atom",
        "force", "energy", "mass", "gravity", "entropy", "plasma",
        "laser", "magnet", "circuit", "gene", "cell", "virus",
        "enzyme", "protein"
    ],
    "TOOL": [
        "hammer", "saw", "drill", "drill", "screw", "nail", "bolt",
        "knife", "blade", "hook", "forge", "wheel", "axle",
        "lever", "axle", "gear", "spring", "joint", "vice", "hook"
    ],
    "COLOUR": [
        "red", "blue", "green", "yellow", "purple", "orange", "pink",
        "brown", "black", "white", "grey", "grey", "violet", "gold",
        "silver", "beige", "azure", "indigo", "violet", "crimson"
    ],
}

words, labels = [], []
for field, wlist in SEMANTIC_FIELDS.items():
    for w in wlist[:20]:
        words.append(w)
        labels.append(field)
labels = np.array(labels)
field_names = list(SEMANTIC_FIELDS.keys())

# ── Load model + build embeddings ─────────────────────────────────────────
model  = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
bert   = model[0].auto_model
layers_bert = bert.encoder.layer

# Probe B: full transformer pass
X_pass_raw = model.encode(words, batch_size=64, show_progress_bar=False, convert_to_numpy=True)
X_pass  = normalize(X_pass_raw, norm="l2")
X_arrow = X_pass * ARROW_MAG

# Probe A: raw E_tok lookup
E_tok     = bert.embeddings.word_embeddings.weight.detach().numpy()
tokenizer = model.tokenizer
token_ids = np.array([tokenizer.encode(w, add_special_tokens=False)[0] for w in words])
X_etok    = normalize(E_tok[token_ids], norm="l2")

# ── Activation matrix (Probe B) ───────────────────────────────────────────
PRIMAL_ROLES = ["W_q", "W_k", "W_v", "W_o", "W_ffn1"]
ROLES_ATT    = PRIMAL_ROLES + ["W_ffn2_read"]
n_roles  = len(ROLES_ATT)
n_layers = len(layers_bert)
col_names = [f"L{i}_{r}" for i in range(n_layers) for r in ROLES_ATT]

weights = {}
for i, layer in enumerate(layers_bert):
    attn = layer.attention.self
    weights[(i, "W_q")]    = attn.query.weight.detach().numpy()
    weights[(i, "W_k")]    = attn.key.weight.detach().numpy()
    weights[(i, "W_v")]    = attn.value.weight.detach().numpy()
    weights[(i, "W_o")]    = layer.attention.output.dense.weight.detach().numpy()
    weights[(i, "W_ffn1")] = layer.intermediate.dense.weight.detach().numpy()
    weights[(i, "W_ffn2")] = layer.output.dense.weight.detach().numpy()

def activation_energy(W, x, role=""):
    if role == "W_ffn2_read":
        W = W.T
    elif role == "W_ffn2" and x.shape[0] != W.shape[1]:
        return 0.0
    return float(np.linalg.norm(W @ x) / (np.linalg.norm(W, "fro") + 1e-9))

N = len(words)
act_matrix = np.zeros((N, n_layers * n_roles))
for n_idx, word_vec in enumerate(X_pass):
    col = 0
    for i in range(n_layers):
        for role in ROLES_ATT:
            W = weights[(i, "W_ffn2")] if role == "W_ffn2_read" else weights[(i, role)]
            act_matrix[n_idx, col] = activation_energy(W, word_vec, role=role)
            col += 1

print(f"Setup complete. Corpus: {N} words | X_pass: {X_pass.shape} | act_matrix: {act_matrix.shape}")

---
## 1 · Utility functions and `build_score_phase`

All IndexScorePhase implementations are wrapped in a single factory:

```python
scores, meta = build_score_phase(name, X, labels, **kwargs)
```

| Phase name | Maps to capability | Method |
|---|---|---|
| `"arrowspace"` | Search | ArrowSpace λ_full via `aspace.search()` |
| `"pca_cosine"` | Analysis (baseline) | Cosine to PCA-2D centroid |
| `"kde"` | Clustering (baseline) | Gaussian KDE density in PCA-2D |
| `"diffmaps"` | Compression (baseline) | Diffusion distance to global centroid |
| `"layer_arrowspace"` | Analysis | ArrowSpace on layer-sliced activation manifold |

In [ ]:
# ── Core utilities ─────────────────────────────────────────────────────────

def norm01(v):
    """Min-max normalise to [0, 1]. Returns zeros if range is zero."""
    lo, hi = v.min(), v.max()
    return (v - lo) / (hi - lo + 1e-12)


def search_elements(aspace, gl, X, alpha=0.5):
    """
    Extract per-item lambda scores from an ArrowSpace instance.
    Uses aspace.search() with the given alpha blend.
    Assigns a high sentinel (1.0) for degenerate zero-lambda items.
    """
    scores = []
    for i, x in enumerate(X):
        try:
            result = aspace.search(x.tolist(), alpha=alpha, top_k=1)
            lam = result[0].score if result else 0.0
        except Exception:
            lam = 0.0
        if lam == 0.0:
            print(f"  [warn] vector {i} has 0.0 lambda — assigning high sentinel")
            lam = 1.0
        scores.append(lam)
    return norm01(np.array(scores, dtype=np.float64))


def cluster_purity(scores, labels, top_k_pct=TOP_K_PCT):
    """Purity of the lowest top_k_pct fraction (low score = in-basin)."""
    k = max(1, int(len(scores) * top_k_pct))
    idx = np.argsort(scores)[:k]
    dominant = pd.Series(labels[idx]).value_counts().iloc[0]
    return dominant / k


def mean_lambda(scores, lambda_ref, top_k_pct=TOP_K_PCT):
    k = max(1, int(len(scores) * top_k_pct))
    idx = np.argsort(scores)[:k]
    return lambda_ref[idx].mean()


def jaccard(scores_a, scores_b, top_k_pct=TOP_K_PCT):
    k = max(1, int(len(scores_a) * top_k_pct))
    set_a = set(np.argsort(scores_a)[:k])
    set_b = set(np.argsort(scores_b)[:k])
    return len(set_a & set_b) / len(set_a | set_b)


# ── Pluggable IndexScorePhase factory ──────────────────────────────────────

def build_score_phase(name, X, labels, graph_params=None, alpha=0.5,
                      layer_idx=None, act_mat=None):
    """
    Factory that returns (scores: np.ndarray, meta: dict) for a named phase.

    Parameters
    ----------
    name         : str  — one of 'arrowspace', 'pca_cosine', 'kde', 'diffmaps',
                          'layer_arrowspace'
    X            : np.ndarray (N, D) — embedding / activation manifold
    labels       : np.ndarray (N,)   — ground-truth semantic field labels
    graph_params : dict  — passed to ArrowSpaceBuilder.build_and_store()
    alpha        : float — blend weight for ArrowSpace search (0=spectral, 1=geometric)
    layer_idx    : int   — layer slice index (only for 'layer_arrowspace')
    act_mat      : np.ndarray — full activation matrix (only for 'layer_arrowspace')

    Returns
    -------
    scores : np.ndarray (N,) normalised to [0, 1]   (lower = more central / in-basin)
    meta   : dict with diagnostic info (purity, aspace handle, etc.)
    """
    gp = graph_params or GRAPH_PARAMS

    if name == "arrowspace":
        aspace, gl = (
            ArrowSpaceBuilder()
            .with_seed(42)
            .with_dims_reduction(enabled=False, eps=None)
            .with_sampling("simple", 1.0)
        ).build_and_store(gp, X.astype(np.float64))
        scores = search_elements(aspace, gl, X, alpha=alpha)
        return scores, {"aspace": aspace, "gl": gl, "alpha": alpha}

    elif name == "pca_cosine":
        # PCA-Cosine: cosine similarity to the PCA-2D centroid
        pca2 = PCA(n_components=2, random_state=42).fit(X)
        X_pca = pca2.transform(X)
        centroid = X_pca.mean(axis=0)
        raw = 1 - cdist(X_pca, centroid[None], metric="cosine").ravel()
        scores = norm01(raw)
        return scores, {"pca": pca2}

    elif name == "kde":
        # KDE: Gaussian kernel density in PCA-2D space
        pca2 = PCA(n_components=2, random_state=42).fit(X)
        X_pca = pca2.transform(X)
        kde = KernelDensity(kernel="gaussian", bandwidth=0.3).fit(X_pca)
        log_density = kde.score_samples(X_pca)
        scores = norm01(np.exp(log_density))
        return scores, {"kde": kde}

    elif name == "diffmaps":
        # DiffMaps: diffusion distance to global mean after one Markov step
        sigma2 = 0.5
        D = pairwise_distances(X, metric="cosine")
        W_diff = np.exp(-D**2 / sigma2)
        P = W_diff / W_diff.sum(axis=1, keepdims=True)
        P2 = P @ P
        diffusion_centroid = P2.mean(axis=0)
        raw = 1 - np.linalg.norm(P2 - diffusion_centroid, axis=1)
        scores = norm01(raw)
        return scores, {"P": P}

    elif name == "layer_arrowspace":
        # Layer-activation ArrowSpace: run on a single layer's activation slice
        assert act_mat is not None and layer_idx is not None
        n_r = len(ROLES_ATT)
        act_slice = act_mat[:, layer_idx * n_r : (layer_idx + 1) * n_r]
        act_norm  = normalize(act_slice, norm="l2")
        aspace_l, gl_l = (
            ArrowSpaceBuilder()
            .with_seed(42)
            .with_dims_reduction(enabled=False, eps=None)
            .with_sampling("simple", 1.0)
        ).build_and_store(gp, act_norm.astype(np.float64))
        scores = search_elements(aspace_l, gl_l, act_norm, alpha=alpha)
        return scores, {"aspace": aspace_l, "gl": gl_l, "layer": layer_idx}

    else:
        raise ValueError(f"Unknown phase name: '{name}'. "
                         "Valid: arrowspace, pca_cosine, kde, diffmaps, layer_arrowspace")


print("build_score_phase() and utilities defined.")

---
## 2 · Build ArrowSpace index (TauModeScore → Search capability)

The ArrowSpace λ-score is the canonical **Search** phase: it computes the synthetic λ
index as a Rayleigh quotient + Dirichlet dispersion blend, acting as a stable
pre-filter that bounds RAG retrieval and mitigates out-of-distribution queries.

We also extract the geometric (`R_geom`) and spectral (`R_spec`) components
for use in the α-sweep (§4) and independence check (§4.1).

In [ ]:
# ── ArrowSpace index on embedding manifold (Probe B — full pass) ──────────
lambda_full, meta_as = build_score_phase("arrowspace", X_arrow, labels, alpha=0.5)
aspace = meta_as["aspace"]
gl     = meta_as["gl"]

# Geometric component (alpha=1.0 → pure geometric Laplacian energy)
R_geom, _ = build_score_phase("arrowspace", X_arrow, labels, alpha=1.0)

# Spectral component — direct Rayleigh quotient to avoid alpha-collapse
# (search() with alpha≈0 degenerates on small corpora; direct RQ is stable)
try:
    L_dense = gl.to_dense().astype(np.float64)
except AttributeError:
    L_dense = gl.toarray().astype(np.float64)

X_n = X_arrow / (np.linalg.norm(X_arrow, axis=1, keepdims=True) + 1e-9)
R_spec_rq = np.array([float(x @ L_dense @ x / (x @ x + 1e-12)) for x in X_n])
R_spec = norm01(R_spec_rq)

print(f"lambda_full   mean={lambda_full.mean():.3f}  std={lambda_full.std():.3f}")
print(f"R_geom        mean={R_geom.mean():.3f}  std={R_geom.std():.3f}")
print(f"R_spec (RQ)   mean={R_spec.mean():.3f}  std={R_spec.std():.3f}")

---
## 3 · Vanilla baseline IndexScorePhases

Three baselines instantiated via `build_score_phase()`:

| Phase | Capability analogue | Rationale |
|---|---|---|
| `pca_cosine` | Analysis — linear projection | Linear centroid geometry; no spectral signal |
| `kde` | Clustering — density estimation | Non-parametric density; field-aware but topology-blind |
| `diffmaps` | Compression — diffusion geometry | Markov diffusion captures local manifold structure |

In [ ]:
v_pca,  meta_pca  = build_score_phase("pca_cosine", X_pass, labels)
v_kde,  meta_kde  = build_score_phase("kde",        X_pass, labels)
v_diff, meta_diff = build_score_phase("diffmaps",   X_pass, labels)

print("Vanilla IndexScorePhases computed.")
print(f"  v_pca   mean={v_pca.mean():.3f}  std={v_pca.std():.3f}")
print(f"  v_kde   mean={v_kde.mean():.3f}  std={v_kde.std():.3f}")
print(f"  v_diff  mean={v_diff.mean():.3f}  std={v_diff.std():.3f}")

---
## 4 · Evaluation harness — purity, mean-λ, Jaccard

Following **P1** (λ is a final score), we compare all phases on:
- **Cluster purity** of the bottom `TOP_K_PCT` basin items
- **Mean λ_full** of the selected basin (lower = deeper in-basin)
- **Jaccard overlap** with ArrowSpace's basin selection

In [ ]:
score_registry = {
    "ArrowSpace (λ_full)": lambda_full,
    "PCA-Cosine":          v_pca,
    "KDE":                 v_kde,
    "DiffMaps":            v_diff,
}

rows = []
for name, scores in score_registry.items():
    rows.append({
        "Phase":         name,
        "Purity":        round(cluster_purity(scores, labels), 3),
        "Mean λ_full":   round(mean_lambda(scores, lambda_full), 3),
        "Jaccard vs AS": round(jaccard(scores, lambda_full), 3)
                         if name != "ArrowSpace (λ_full)" else 1.0,
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))
df_results.to_csv(OUTPUT_DIR / "comparison_results.csv", index=False)

### 4.1 · Independence check — R_spec vs vanilla scores (P6)

We verify that `R_spec` (direct Rayleigh quotient) is not a disguised copy of any
vanilla score by plotting scatter plots with Pearson ρ.
Low |ρ| confirms spectral augmentation adds **orthogonal information**.

In [ ]:
fig_indep = make_subplots(rows=1, cols=3,
    subplot_titles=["R_spec vs PCA-Cosine", "R_spec vs KDE", "R_spec vs DiffMaps"])

vanilla_pairs = [("PCA-Cosine", v_pca), ("KDE", v_kde), ("DiffMaps", v_diff)]
for col_idx, (vname, v) in enumerate(vanilla_pairs, start=1):
    rho, _ = pearsonr(R_spec, v)
    fig_indep.add_trace(go.Scatter(
        x=v, y=R_spec,
        mode="markers",
        text=[f"{w} ({l})" for w, l in zip(words, labels)],
        marker=dict(
            color=[FIELD_COLOURS.get(l, "#aaa") for l in labels],
            size=7,
        ),
        showlegend=False,
        name=vname,
    ), row=1, col=col_idx)
    fig_indep.add_annotation(
        xref=f"x{col_idx}", yref=f"y{col_idx}",
        x=0.95, y=0.95, xanchor="right", yanchor="top",
        text=f"ρ = {rho:.3f}",
        showarrow=False, font=dict(size=12),
    )

fig_indep.update_xaxes(title_text="Vanilla score v(x)")
fig_indep.update_yaxes(title_text="R_spec(x)", col=1)
fig_indep.update_layout(
    height=380,
    title_text="Independence check: R_spec vs Vanilla IndexScorePhases",
    font_family="monospace", title_font_size=14,
)
fig_indep.write_image(OUTPUT_DIR / "fig_06C_independence.png", scale=2)
fig_indep.show()
print("Saved fig_06C_independence.png")

### 4.2 · α-sweep — spectral augmentation (P3 + P5)

$$\text{aug}_{\alpha}(x) = \alpha \cdot v(x) + (1-\alpha) \cdot R_{\text{spec}}(x)$$

We sweep `α ∈ [0, 1]` for each vanilla phase and track purity and mean-λ,
reporting at which α each method recovers ArrowSpace-level basin coherence.

In [ ]:
alphas = np.linspace(0, 1, ALPHA_STEPS)
vanilla_methods = {"PCA-Cosine": v_pca, "KDE": v_kde, "DiffMaps": v_diff}

sweep_rows = []
for method_name, v in vanilla_methods.items():
    for alpha in alphas:
        aug = alpha * v + (1 - alpha) * R_spec
        sweep_rows.append({
            "Phase":      method_name,
            "alpha":      round(float(alpha), 2),
            "Purity":     cluster_purity(aug, labels),
            "MeanLambda": mean_lambda(aug, lambda_full),
        })

df_sweep = pd.DataFrame(sweep_rows)
df_sweep.to_csv(OUTPUT_DIR / "alpha_sweep.csv", index=False)

fig_sweep = make_subplots(rows=1, cols=2,
    subplot_titles=["Cluster Purity vs α", "Mean λ_full vs α"])

colors = px.colors.qualitative.Set2
for m_idx, method in enumerate(vanilla_methods):
    sub = df_sweep[df_sweep["Phase"] == method]
    fig_sweep.add_trace(go.Scatter(
        x=sub["alpha"], y=sub["Purity"],
        mode="lines+markers", name=method,
        line=dict(color=colors[m_idx])), row=1, col=1)
    fig_sweep.add_trace(go.Scatter(
        x=sub["alpha"], y=sub["MeanLambda"],
        mode="lines+markers", name=method, showlegend=False,
        line=dict(color=colors[m_idx], dash="dot")), row=1, col=2)

# ArrowSpace baseline reference lines
as_purity = cluster_purity(lambda_full, labels)
as_ml     = mean_lambda(lambda_full, lambda_full)
for col_idx, yval in [(1, as_purity), (2, as_ml)]:
    fig_sweep.add_hline(
        y=yval, line_dash="dash", line_color="black",
        annotation_text="ArrowSpace λ_full", row=1, col=col_idx)

fig_sweep.update_xaxes(title_text="α  (1 = pure vanilla,  0 = pure R_spec)")
fig_sweep.update_layout(
    height=420,
    title_text="α Sweeps — Spectral Augmentation of Vanilla Phases",
    font_family="monospace", title_font_size=14,
)
fig_sweep.write_image(OUTPUT_DIR / "fig_06C_alpha_sweep.png", scale=2)
fig_sweep.show()
print("Saved fig_06C_alpha_sweep.png")

---
## 5 · Layer-activation-aware ArrowSpace (Analysis capability)

We instantiate the `layer_arrowspace` phase for each of the 6 transformer layers.
This reveals **which layer's activation pattern is most semantically coherent**,
implementing the `FeatureSpectralScore` Analysis track: the activation manifold
at each layer is treated as a separate feature space and wired via ArrowSpace.

In [ ]:
layer_probe_rows = []
for l_idx in range(n_layers):
    lf_layer, meta_layer = build_score_phase(
        "layer_arrowspace", X_pass, labels,
        layer_idx=l_idx, act_mat=act_matrix, alpha=0.5,
    )
    layer_probe_rows.append({
        "Layer":         f"Layer {l_idx}",
        "Purity":        round(cluster_purity(lf_layer, labels), 3),
        "Mean λ_full":   round(mean_lambda(lf_layer, lambda_full), 3),
        "Jaccard vs AS": round(jaccard(lf_layer, lambda_full), 3),
    })

df_layer = pd.DataFrame(layer_probe_rows)
print(df_layer.to_string(index=False))
df_layer.to_csv(OUTPUT_DIR / "layer_probe_results.csv", index=False)

### 5.1 · Per-field score distribution — visualisation

In [ ]:
# Per-field mean score for each phase — grouped bar chart
all_scores = {
    "AS λ_full":  lambda_full,
    "PCA-Cosine": v_pca,
    "KDE":        v_kde,
    "DiffMaps":   v_diff,
}

fig_bar = go.Figure()
bar_colors = ["rgb(102, 197, 204)", "rgb(246, 207, 113)",
              "rgb(248, 156, 116)", "rgb(220, 176, 242)"]

for (pname, sc), bc in zip(all_scores.items(), bar_colors):
    per_field = [
        sc[labels == f].mean() for f in field_names
    ]
    fig_bar.add_trace(go.Bar(
        name=pname,
        x=field_names,
        y=per_field,
        marker_color=bc,
    ))

fig_bar.update_layout(
    barmode="group",
    height=450,
    title_text="Per-field mean score — all IndexScorePhases",
    yaxis_title="Mean score (lower = more central)",
    font_family="monospace", title_font_size=14,
)
fig_bar.write_image(OUTPUT_DIR / "fig_06C_per_field_scores.png", scale=2)
fig_bar.show()
print("Saved fig_06C_per_field_scores.png")

---
## 6 · Capability interpretation — Graph Wiring mapping

This section explicitly maps each `IndexScorePhase` to the corresponding
ArrowSpace Graph Wiring capability, explains the observable evidence from
the evaluation results above, and documents the pluggability pattern.

### 6.1 · Score-to-capability mapping

| Capability | Phase | Score used | What the score measures | Evidence from §4 |
|---|---|---|---|---|
| **Search** | `TauModeScore` (ArrowSpace λ_full) | `lambda_full` | Rayleigh quotient + Dirichlet dispersion on the k-NN Laplacian; items in smooth semantic basins receive low λ | Jaccard = 1.0 by definition; mean-λ baseline for all comparisons |
| **Analysis** | `FeatureSpectralScore` (layer ArrowSpace) | `lf_layer[l]` | ArrowSpace wired on the activation manifold at layer `l`; reveals which transformer layer produces the most semantically coherent feature-space topology | Layer purity profile in §5; divergence from embedding-space λ_full |
| **Clustering** | `FiedlerVectorScore` (approximated by KDE) | `v_kde` | KDE density in PCA-2D approximates the non-parametric density used to identify natural groupings; Fiedler vector would give hard bipartite splits | High purity at α=1 (pure KDE); Jaccard vs AS shows distinct basin selections |
| **Classification** | `BoundaryEnergyScore` (approximated by R_spec) | `R_spec` | Direct Rayleigh quotient on the Laplacian eigenbasis; high R_spec → item near spectral boundary → candidate for novel class | Independence check §4.1: low Pearson ρ vs all vanilla scores confirms orthogonal signal |
| **Compression** | `EnergyDiffusionScore` (approximated by DiffMaps) | `v_diff` | Diffusion distance to global centroid after one Markov step; items far from the diffusion centroid are high-information and should be retained | DiffMaps Jaccard vs AS reveals which items ArrowSpace and diffusion geometry agree are redundant |

> **Key architectural implication**: all five capabilities share the same upstream
> pipeline (`clustering → sampling → graph Laplacian`). Only the terminal `IndexScorePhase`
> differs. The `build_score_phase(name, X, ...)` factory above is the concrete
> implementation of this pluggability contract.

### 6.2 · α-sweep interpretation

The α-sweep in §4.2 tests **Principle P3** (spectral-only augmentation):

$$\text{aug}_{\alpha}(x) = \alpha \cdot v(x) + (1-\alpha) \cdot R_{\text{spec}}(x)$$

- At **α = 1** (pure vanilla): each method uses only its own geometric signal.
- At **α = 0** (pure spectral): all methods collapse to `R_spec` — the Laplacian eigenbasis projection.
- **Crossover point**: the α at which a vanilla phase first matches ArrowSpace purity
  indicates how much spectral information was missing from its original score.
  A crossover near α ≈ 0.5 means the method benefits significantly from spectral augmentation.

If `R_spec` is degenerate (flat zero), the α-sweep will be flat and the crossover
will not appear. In that case, the direct Rayleigh quotient version of `R_spec`
(as built in §2 above) should be used instead of the `aspace.search(alpha=0)` version.

### 6.3 · Summary table

In [ ]:
# Consolidated capability summary with evaluation metrics
capability_map = [
    {
        "Capability":    "Search",
        "Phase":         "TauModeScore",
        "Variable":      "lambda_full",
        "Purity":        round(cluster_purity(lambda_full, labels), 3),
        "Mean λ_full":   round(mean_lambda(lambda_full, lambda_full), 3),
        "Jaccard vs AS": 1.0,
    },
    {
        "Capability":    "Analysis",
        "Phase":         "FeatureSpectralScore (best layer)",
        "Variable":      "lf_layer[best]",
        "Purity":        df_layer["Purity"].max(),
        "Mean λ_full":   df_layer.loc[df_layer["Purity"].idxmax(), "Mean λ_full"],
        "Jaccard vs AS": df_layer["Jaccard vs AS"].max(),
    },
    {
        "Capability":    "Clustering",
        "Phase":         "FiedlerVectorScore (≈ KDE)",
        "Variable":      "v_kde",
        "Purity":        round(cluster_purity(v_kde, labels), 3),
        "Mean λ_full":   round(mean_lambda(v_kde, lambda_full), 3),
        "Jaccard vs AS": round(jaccard(v_kde, lambda_full), 3),
    },
    {
        "Capability":    "Classification",
        "Phase":         "BoundaryEnergyScore (≈ R_spec)",
        "Variable":      "R_spec",
        "Purity":        round(cluster_purity(R_spec, labels), 3),
        "Mean λ_full":   round(mean_lambda(R_spec, lambda_full), 3),
        "Jaccard vs AS": round(jaccard(R_spec, lambda_full), 3),
    },
    {
        "Capability":    "Compression",
        "Phase":         "EnergyDiffusionScore (≈ DiffMaps)",
        "Variable":      "v_diff",
        "Purity":        round(cluster_purity(v_diff, labels), 3),
        "Mean λ_full":   round(mean_lambda(v_diff, lambda_full), 3),
        "Jaccard vs AS": round(jaccard(v_diff, lambda_full), 3),
    },
]

df_capability = pd.DataFrame(capability_map)
print(df_capability.to_string(index=False))
df_capability.to_csv(OUTPUT_DIR / "capability_mapping.csv", index=False)
print("\nSaved capability_mapping.csv")